# 🦉 Strixlibe × Google Colab GPU LLM Server (Ollama + hf-transfer 爆速版)
自律型ペンテストエージェント「Strixlibe」向けに、Google Colab（**L4 / T4 / G4 GPU**）上で、Python/CUDAのメモリ衝突エラーを起こさない **Ollama** を使ってオープンソースモデル（DeepSeek-R1 14B/32B、Qwen2.5-Coder 14B/7B 等）をホストし、Cloudflare Tunnel 経由で安全に連携するノートブックです。

### ⚡ メリット & 高速化機能
- **Python/PyTorch 依存ゼロ**: ネイティブ C++ バイナリで動くため、CUDA メモリ初期化エラー（OOM）や Python 競合が一切起きません。
- **hf-transfer による爆速ダウンロード**: Hugging Face公式のRust製並列ダウンローダー（`hf-transfer`）を標準搭載。Colabの数Gbps帯域をフル活用し、DL時間を大幅に短縮します。
- **ダウンロード進捗が 100% 見える**: パーセント表示（%）とダウンロード速度がリアルタイムで画面に表示されます。
- **OpenAI 互換 API 完全対応**: `https://<tunnel-url>/v1` で Strixlibe から呼び出せます。
- **CU保護**: 20分間リクエストが途絶えると自動でランタイムを切断・停止


### 1. 設定項目（フォーム）

In [ ]:
#@title 🚀 サーバー基本設定 { run: "auto" }
MODEL_NAME = "deepseek-r1:14b" #@param ["deepseek-r1:14b", "qwen2.5-coder:14b", "qwen2.5-coder:7b", "deepseek-r1:32b", "qwen2.5-coder:32b", "hf.co/bartowski/DeepSeek-R1-Distill-Qwen-14B-GGUF:Q4_K_M"] {allow-input: true}
HF_TOKEN = "" #@param {type:"string"}
ENABLE_HF_TRANSFER = True #@param {type:"boolean"}
DISPATCHER_URL = "https://strix-dispatcher-2acg5ejyka-uw.a.run.app" #@param {type:"string"}
DISPATCHER_API_KEY = "" #@param {type:"string"}
SLACK_WEBHOOK_URL = "" #@param {type:"string"}
IDLE_TIMEOUT_MINUTES = 20 #@param {type:"integer"}
TARGET_ENV = "all" #@param ["all", "staging", "production"]


### 2. Ollama / cloudflared / hf-transfer のセットアップ（約30秒）


In [ ]:
# GPUの確認
!nvidia-smi

# 過去の残存プロセスを完全停止
!pkill -9 -f ollama || true
!pkill -9 -f cloudflared || true

# 必要なシステムパッケージの導入（Ollamaのインストーラ要件）
!apt-get update -qq && apt-get install -y -qq zstd pciutils

# Hugging Face 高速並列ダウンローダー (hf-transfer) の導入
!pip install -q -U "huggingface_hub[hf_transfer]" hf-transfer

import os
if ENABLE_HF_TRANSFER:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    print("[+] hf-transfer (Rust parallel accelerator) enabled!")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    !huggingface-cli login --token $HF_TOKEN --add-to-git-credential || true
    print("[+] Hugging Face token registered.")

# Ollama のインストール（公式スクリプト）
!curl -fsSL https://ollama.com/install.sh | sh

# cloudflared のダウンロード
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# インストール確認
!/usr/local/bin/ollama --version || /usr/bin/ollama --version || ollama --version
print("\n[+] Ollama, cloudflared & hf-transfer installed successfully!")


### 3. Ollama サーバーと Cloudflare Tunnel の起動

In [ ]:
import os
import subprocess
import time
import re
import requests
import shutil

# ollama バイナリの絶対パスを自動検出
ollama_bin = shutil.which("ollama")
if not ollama_bin:
    for candidate in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/bin/ollama"]:
        if os.path.exists(candidate):
            ollama_bin = candidate
            break

if not ollama_bin:
    raise FileNotFoundError("Ollama binary not found. Please make sure Cell 2 finished successfully.")

print(f"[*] Found Ollama binary at: {ollama_bin}")

# Ollama の環境変数設定（全IP許可・CORS許可）
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"

# Ollama サーバーをバックグラウンド起動
print("[*] Starting Ollama server...")
ollama_log = open("ollama.log", "w")
ollama_proc = subprocess.Popen([ollama_bin, "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)

# Ollama の起動待機
while True:
    try:
        r = requests.get("http://127.0.0.1:11434/", timeout=1)
        if r.status_code == 200:
            print("[+] Ollama server is running!")
            break
    except:
        pass
    time.sleep(1)

# Cloudflare Tunnel の起動（HTTP/2 プロトコル固定・Ollama 403防止）
print("[*] Establishing Cloudflare Tunnel (HTTP/2 mode)...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--protocol", "http2", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
while True:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print(f"\n🎉 ==========================================================")
        print(f"[+] Cloudflare Tunnel established successfully!")
        print(f"[+] Public URL: {tunnel_url}")
        print(f"[+] OpenAI API Base: {tunnel_url}/v1")
        print(f"==========================================================\n")
        break

api_base = f"{tunnel_url}/v1"

### 4. モデルのダウンロード（hf-transfer 高速化対応）


In [ ]:
import os
import re
import subprocess

print(f"[*] Preparing model: {MODEL_NAME} ...")

# Hugging Face Hub (hf.co / repo) の場合
if MODEL_NAME.startswith("hf.co/") or ("/" in MODEL_NAME and not MODEL_NAME.startswith("http")):
    clean_hf_name = MODEL_NAME.replace("hf.co/", "")
    print(f"[*] Downloading Hugging Face model via hf-transfer: {clean_hf_name}")
    
    # GGUF ファイル名指定がある場合 (repo:filename または repo)
    if ":" in clean_hf_name:
        repo_id, filename = clean_hf_name.split(":", 1)
        if not filename.endswith(".gguf"):
            filename = f"{filename}.gguf"
    else:
        repo_id = clean_hf_name
        filename = None
    
    from huggingface_hub import hf_hub_download, list_repo_files
    
    if not filename:
        print(f"[*] Searching GGUF files in repo: {repo_id} ...")
        files = list_repo_files(repo_id)
        gguf_files = [f for f in files if f.endswith(".gguf")]
        # Q4_K_M または最初のGGUFを優先
        preferred = [f for f in gguf_files if "Q4_K_M" in f or "q4_k_m" in f]
        filename = preferred[0] if preferred else gguf_files[0]
        print(f"[*] Selected file: {filename}")
    
    print(f"[*] Downloading {filename} from {repo_id} using Rust hf-transfer ...")
    local_model_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir="/content/models",
        local_dir_use_symlinks=False
    )
    print(f"[+] Model downloaded to: {local_model_path}")
    
    # Ollama 用 Modelfile の生成とインポート
    alias_name = re.sub(r"[^a-zA-Z0-9_.-]", "-", repo_id.split("/")[-1]).lower()
    modelfile_content = f"FROM {local_model_path}\n"
    with open("/content/Modelfile", "w") as f:
        f.write(modelfile_content)
    
    print(f"[*] Creating Ollama model: {alias_name} ...")
    subprocess.run([ollama_bin, "create", alias_name, "-f", "/content/Modelfile"], check=True)
    MODEL_NAME = alias_name
    print(f"[+] Registered Hugging Face model as Ollama model: {MODEL_NAME}")
else:
    # 通常の Ollama レジストリモデル
    print(f"[*] Pulling Ollama registry model: {MODEL_NAME} (approx 1-3 minutes) ...")
    !{ollama_bin} pull {MODEL_NAME}

print(f"\n[+] Model {MODEL_NAME} is ready and loaded in GPU!")


### 5. 疎通テスト（初回ウォームアップ対応・タイムアウト120秒）

In [ ]:
print(f"[*] Sending test prompt to: {api_base}/chat/completions ...")
print("[*] (Note: First request loads weights into VRAM, please wait ~30-60s)")

test_payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Strixペンテストエージェントからの疎通確認テストです。接続が正常であることを、日本語で1〜2文で応答してください。"}
    ],
    "max_tokens": 150,
    "temperature": 0.6
}

try:
    resp = requests.post(f"{api_base}/chat/completions", json=test_payload, timeout=120)
    if resp.status_code == 200:
        result = resp.json()
        reply = result["choices"][0]["message"]["content"]
        print("\n[+] 推論テスト成功！モデルからの返答:")
        print("--------------------------------------------------")
        print(reply.strip())
        print("--------------------------------------------------")
    else:
        print(f"[-] テスト失敗: HTTP {resp.status_code} - {resp.text}")
except Exception as e:
    print(f"[-] テスト例外: {e}")

### 6. Dispatcher 自動同期 & Slack 通知

In [ ]:
# Dispatcher へのセッション登録
if DISPATCHER_URL and DISPATCHER_API_KEY:
    try:
        reg_payload = {
            "provider": "colab-ollama",
            "base_url": api_base,
            "model_name": MODEL_NAME,
            "env_target": TARGET_ENV,
            "ttl_minutes": 120
        }
        headers = {"X-API-Key": DISPATCHER_API_KEY, "Content-Type": "application/json"}
        reg_res = requests.post(f"{DISPATCHER_URL}/api/v1/llm-session", json=reg_payload, headers=headers, timeout=10)
        print(f"[+] Registered to Dispatcher: status {reg_res.status_code}")
    except Exception as e:
        print(f"[-] Dispatcher registration skipped or failed: {e}")
else:
    print("[*] Dispatcher 未設定のため同期をスキップしました（手動接続可能モード）。")

# Slack 通知
if SLACK_WEBHOOK_URL:
    try:
        msg = (
            f"🚀 *Strixlibe Colab GPU LLM Server is ONLINE!*\n"
            f"• *Engine*: Ollama\n"
            f"• *Model*: `{MODEL_NAME}`\n"
            f"• *Endpoint*: `{api_base}`\n"
            f"• *Env*: `{TARGET_ENV}`\n"
            f"• *Watchdog*: {IDLE_TIMEOUT_MINUTES} mins auto-kill\n"
            f"💡 Strixlibe will automatically route scans to this model."
        )
        requests.post(SLACK_WEBHOOK_URL, json={"text": msg}, timeout=5)
        print("[+] Slack notification sent.")
    except Exception as e:
        print(f"[-] Slack notification failed: {e}")

### 7. CU保護ウォッチドッグ（バックグラウンド監視・自動シャットダウン）

In [ ]:
import threading
from google.colab import runtime

def watchdog_worker():
    print(f"[*] Watchdog background worker started. Auto-terminate after {IDLE_TIMEOUT_MINUTES}m inactivity.")
    last_seen_time = time.time()
    idle_timeout_seconds = IDLE_TIMEOUT_MINUTES * 60
    
    while True:
        time.sleep(30)
        elapsed = time.time() - last_seen_time
        if elapsed > idle_timeout_seconds:
            print(f"[!] Inactivity threshold ({IDLE_TIMEOUT_MINUTES} min) exceeded. Terminating runtime to preserve Compute Units.")
            if SLACK_WEBHOOK_URL:
                try:
                    requests.post(SLACK_WEBHOOK_URL, json={"text": "🛑 *Colab LLM Server disconnected* (Idle timeout). Auto-fallback to OpenCode(Zen) engaged."}, timeout=5)
                except:
                    pass
            runtime.unassign()
            break

# バックグラウンドスレッドとして起動
t = threading.Thread(target=watchdog_worker, daemon=True)
t.start()
print("[OK] All systems operational! Tunnel is active and watchdog is monitoring.")